# Análise do Dataset AxonData Face Anti-Spoofing

## Objetivo

Este notebook realiza a caracterização da versão pública do dataset
**AxonData Face Anti-Spoofing**, descarregada através do Hugging Face.

Esta versão do notebook **não utiliza `pandas` nem `numpy`**, para evitar o
bloqueio provocado pela política de Controlo de Aplicações do Windows.

A análise inclui:

- estrutura de diretórios;
- número e tipo de ficheiros;
- distribuição `bona_fide` vs. `spoof`;
- tipos de ataques;
- imagens vs. vídeos;
- resoluções das imagens;
- propriedades dos vídeos, caso `opencv-python` esteja disponível;
- exportação de um relatório CSV;
- resumo final para utilização na dissertação.

## 1. Imports e configuração

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean
from PIL import Image
import csv
import math

DATASET_PATH = Path(
    r"C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Datasets\AxonData_FAS"
)

assert DATASET_PATH.exists(), f"Dataset não encontrado em: {DATASET_PATH}"

print("Dataset encontrado em:")
print(DATASET_PATH)

Dataset encontrado em:
C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Datasets\AxonData_FAS


## 2. Estrutura principal

In [2]:
categories = [
    path
    for path in sorted(DATASET_PATH.iterdir())
    if path.is_dir() and not path.name.startswith(".")
]

print("Categorias encontradas:\n")

for category in categories:
    print("-", category.name)

Categorias encontradas:

- 3D_paper_mask
- Cutout_attacks
- Latex_mask
- Replay_display_attacks
- Replay_mobile_attacks
- Selfies
- Silicone_mask
- Textile 3D Face Mask Attack Sample
- Wrapped_3D_paper_mask


## 3. Tipos de ficheiro

In [3]:
MEDIA_EXTENSIONS = {".jpg", ".jpeg", ".png", ".mp4", ".mov"}

extensions = Counter()

for file in DATASET_PATH.rglob("*"):
    if file.is_file() and ".cache" not in file.parts:
        extensions[file.suffix.lower()] += 1

print("TIPOS DE FICHEIRO\n")

for ext, count in extensions.most_common():
    print(f"{ext or '[sem extensão]'}: {count}")

TIPOS DE FICHEIRO

.mp4: 84
.mov: 36
.jpg: 29
[sem extensão]: 1
.md: 1


In [4]:
print("TIPOS DE FICHEIRO POR CATEGORIA\n")

for category_path in categories:
    counts = Counter(
        file.suffix.lower()
        for file in category_path.rglob("*")
        if file.is_file()
    )

    print(category_path.name)

    for ext, count in counts.items():
        print(f"  {ext or '[sem extensão]'}: {count}")

    print()

TIPOS DE FICHEIRO POR CATEGORIA

3D_paper_mask
  .mp4: 36

Cutout_attacks
  .mov: 8
  .mp4: 7

Latex_mask
  .mp4: 5
  .mov: 5

Replay_display_attacks
  .jpg: 5
  .mp4: 3
  .mov: 2

Replay_mobile_attacks
  .mp4: 10

Selfies
  .jpg: 24

Silicone_mask
  .mp4: 5
  .mov: 6

Textile 3D Face Mask Attack Sample
  .mov: 12
  .mp4: 11

Wrapped_3D_paper_mask
  .mp4: 7
  .mov: 3



## 4. Construção do registo das amostras

A categoria `Selfies` é tratada como `bona_fide`. As restantes categorias são
tratadas como `spoof`.

Não são inferidos sujeitos nem dispositivos sem suporte explícito na estrutura
ou documentação do dataset.

In [5]:
attack_mapping = {
    "Selfies": "none",
    "3D_paper_mask": "3d_paper_mask",
    "Cutout_attacks": "cutout",
    "Latex_mask": "latex_mask",
    "Replay_display_attacks": "replay_display",
    "Replay_mobile_attacks": "replay_mobile",
    "Silicone_mask": "silicone_mask",
    "Textile 3D Face Mask Attack Sample": "textile_3d_mask",
    "Wrapped_3D_paper_mask": "wrapped_3d_paper_mask",
}

records = []

for category_path in categories:
    category = category_path.name

    for file in category_path.rglob("*"):
        if not file.is_file():
            continue

        extension = file.suffix.lower()

        if extension not in MEDIA_EXTENSIONS:
            continue

        label = "bona_fide" if category == "Selfies" else "spoof"
        class_id = 0 if label == "bona_fide" else 1
        media_type = "video" if extension in {".mp4", ".mov"} else "image"

        records.append({
            "filepath": str(file),
            "relative_path": str(file.relative_to(DATASET_PATH)),
            "filename": file.name,
            "category": category,
            "label": label,
            "class_id": class_id,
            "attack_type": attack_mapping.get(category, "unknown"),
            "media_type": media_type,
            "extension": extension,
            "file_size_mb": round(file.stat().st_size / (1024 ** 2), 3),
            "width": None,
            "height": None,
            "fps": None,
            "frame_count": None,
            "duration_seconds": None,
        })

print("Número total de amostras:", len(records))

print("\nPrimeiras 5 amostras:\n")
for row in records[:5]:
    print(row)

Número total de amostras: 149

Primeiras 5 amostras:

{'filepath': 'C:\\Users\\luisc\\Desktop\\Faculdade\\Cadeiras\\TESE\\Datasets\\AxonData_FAS\\3D_paper_mask\\133.mp4', 'relative_path': '3D_paper_mask\\133.mp4', 'filename': '133.mp4', 'category': '3D_paper_mask', 'label': 'spoof', 'class_id': 1, 'attack_type': '3d_paper_mask', 'media_type': 'video', 'extension': '.mp4', 'file_size_mb': 9.397, 'width': None, 'height': None, 'fps': None, 'frame_count': None, 'duration_seconds': None}
{'filepath': 'C:\\Users\\luisc\\Desktop\\Faculdade\\Cadeiras\\TESE\\Datasets\\AxonData_FAS\\3D_paper_mask\\134.mp4', 'relative_path': '3D_paper_mask\\134.mp4', 'filename': '134.mp4', 'category': '3D_paper_mask', 'label': 'spoof', 'class_id': 1, 'attack_type': '3d_paper_mask', 'media_type': 'video', 'extension': '.mp4', 'file_size_mb': 9.983, 'width': None, 'height': None, 'fps': None, 'frame_count': None, 'duration_seconds': None}
{'filepath': 'C:\\Users\\luisc\\Desktop\\Faculdade\\Cadeiras\\TESE\\Datasets

## 5. Validação

In [6]:
paths = [row["filepath"] for row in records]
filenames = [row["filename"] for row in records]

print("Linhas:", len(records))
print("Caminhos duplicados:", len(paths) - len(set(paths)))
print("Filenames duplicados:", len(filenames) - len(set(filenames)))

Linhas: 149
Caminhos duplicados: 0
Filenames duplicados: 6


## 6. Distribuição das classes

In [7]:
class_counts = Counter(row["label"] for row in records)

print("DISTRIBUIÇÃO DAS CLASSES\n")

for label, count in class_counts.most_common():
    percentage = count / len(records) * 100
    print(f"{label}: {count} ({percentage:.2f}%)")

DISTRIBUIÇÃO DAS CLASSES

spoof: 125 (83.89%)
bona_fide: 24 (16.11%)


## 7. Distribuição por tipo de ataque

In [8]:
attack_counts = Counter(row["attack_type"] for row in records)

print("TIPOS DE ATAQUE\n")

for attack, count in attack_counts.most_common():
    percentage = count / len(records) * 100
    print(f"{attack}: {count} ({percentage:.2f}%)")

TIPOS DE ATAQUE

3d_paper_mask: 36 (24.16%)
none: 24 (16.11%)
textile_3d_mask: 23 (15.44%)
cutout: 15 (10.07%)
silicone_mask: 11 (7.38%)
latex_mask: 10 (6.71%)
replay_display: 10 (6.71%)
replay_mobile: 10 (6.71%)
wrapped_3d_paper_mask: 10 (6.71%)


## 8. Imagens vs. vídeos

In [9]:
media_counts = Counter(row["media_type"] for row in records)

print("TIPO DE MEDIA\n")

for media_type, count in media_counts.most_common():
    print(f"{media_type}: {count}")

print("\nPOR CATEGORIA\n")

for category in sorted(set(row["category"] for row in records)):
    subset = [row for row in records if row["category"] == category]
    counts = Counter(row["media_type"] for row in subset)

    print(category)
    for media_type, count in counts.items():
        print(f"  {media_type}: {count}")

TIPO DE MEDIA

video: 120
image: 29

POR CATEGORIA

3D_paper_mask
  video: 36
Cutout_attacks
  video: 15
Latex_mask
  video: 10
Replay_display_attacks
  image: 5
  video: 5
Replay_mobile_attacks
  video: 10
Selfies
  image: 24
Silicone_mask
  video: 11
Textile 3D Face Mask Attack Sample
  video: 23
Wrapped_3D_paper_mask
  video: 10


## 9. Resoluções das imagens

In [10]:
image_errors = []

for row in records:
    if row["media_type"] != "image":
        continue

    try:
        with Image.open(row["filepath"]) as img:
            row["width"], row["height"] = img.size
    except Exception as exc:
        image_errors.append((row["filepath"], str(exc)))

images = [row for row in records if row["media_type"] == "image"]

print("Imagens analisadas:", len(images))
print("Erros:", len(image_errors))

if images:
    widths = [row["width"] for row in images if row["width"] is not None]
    heights = [row["height"] for row in images if row["height"] is not None]

    print("\nLargura mínima:", min(widths))
    print("Largura máxima:", max(widths))
    print("Altura mínima:", min(heights))
    print("Altura máxima:", max(heights))

    resolution_counts = Counter(
        f'{row["width"]}x{row["height"]}'
        for row in images
        if row["width"] is not None and row["height"] is not None
    )

    print("\nNúmero de resoluções distintas:", len(resolution_counts))
    print("\nResoluções mais frequentes:")

    for resolution, count in resolution_counts.most_common(20):
        print(f"{resolution}: {count}")

Imagens analisadas: 29
Erros: 0

Largura mínima: 811
Largura máxima: 3264
Altura mínima: 1080
Altura máxima: 2448

Número de resoluções distintas: 8

Resoluções mais frequentes:
1000x1333: 21
1000x1336: 2
1000x2222: 1
1000x1339: 1
811x1080: 1
1080x1920: 1
1000x1783: 1
3264x2448: 1


## 10. Propriedades dos vídeos

Esta secção é opcional e utiliza `opencv-python`.

Caso `cv2` ainda não esteja instalado, executar no terminal:

`python -m pip install opencv-python`

O resto do notebook funciona mesmo sem OpenCV.

In [11]:
try:
    import cv2
    CV2_AVAILABLE = True
    print("OpenCV disponível:", cv2.__version__)
except ImportError:
    CV2_AVAILABLE = False
    print("OpenCV não está instalado.")
    print("Instalar com: python -m pip install opencv-python")

OpenCV disponível: 5.0.0


In [12]:
video_errors = []

if CV2_AVAILABLE:
    for row in records:
        if row["media_type"] != "video":
            continue

        cap = cv2.VideoCapture(row["filepath"])

        if not cap.isOpened():
            video_errors.append(
                (row["filepath"], "Não foi possível abrir o vídeo.")
            )
            cap.release()
            continue

        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

        row["width"] = int(round(width)) if width else None
        row["height"] = int(round(height)) if height else None
        row["fps"] = round(float(fps), 3) if fps else None
        row["frame_count"] = int(round(frame_count)) if frame_count else None

        if fps and fps > 0 and frame_count:
            row["duration_seconds"] = round(float(frame_count / fps), 3)

        cap.release()

    videos = [row for row in records if row["media_type"] == "video"]

    print("Vídeos analisados:", len(videos))
    print("Erros:", len(video_errors))

Vídeos analisados: 120
Erros: 0


In [13]:
if CV2_AVAILABLE:
    videos_ok = [
        row for row in records
        if row["media_type"] == "video"
        and row["width"] is not None
        and row["height"] is not None
    ]

    if videos_ok:
        widths = [row["width"] for row in videos_ok]
        heights = [row["height"] for row in videos_ok]

        print("RESOLUÇÃO DOS VÍDEOS\n")
        print("Largura mínima:", min(widths))
        print("Largura máxima:", max(widths))
        print("Altura mínima:", min(heights))
        print("Altura máxima:", max(heights))

        resolution_counts = Counter(
            f'{row["width"]}x{row["height"]}'
            for row in videos_ok
        )

        print("\nResoluções distintas:", len(resolution_counts))
        print("\nResoluções mais frequentes:")

        for resolution, count in resolution_counts.most_common(20):
            print(f"{resolution}: {count}")

        fps_values = [
            row["fps"] for row in videos_ok
            if row["fps"] is not None
        ]

        durations = [
            row["duration_seconds"] for row in videos_ok
            if row["duration_seconds"] is not None
        ]

        frames = [
            row["frame_count"] for row in videos_ok
            if row["frame_count"] is not None
        ]

        if fps_values:
            print(f"\nFPS mínimo: {min(fps_values):.2f}")
            print(f"FPS máximo: {max(fps_values):.2f}")
            print(f"FPS médio: {mean(fps_values):.2f}")

        if durations:
            print(f"\nDuração mínima: {min(durations):.2f} s")
            print(f"Duração máxima: {max(durations):.2f} s")
            print(f"Duração média: {mean(durations):.2f} s")

        if frames:
            print(f"\nFrames mínimos: {min(frames)}")
            print(f"Frames máximos: {max(frames)}")
            print(f"Frames médios: {mean(frames):.2f}")

RESOLUÇÃO DOS VÍDEOS

Largura mínima: 1080
Largura máxima: 2160
Altura mínima: 1920
Altura máxima: 3840

Resoluções distintas: 2

Resoluções mais frequentes:
2160x3840: 74
1080x1920: 46

FPS mínimo: 29.99
FPS máximo: 60.05
FPS médio: 31.29

Duração mínima: 3.40 s
Duração máxima: 18.73 s
Duração média: 9.31 s

Frames mínimos: 102
Frames máximos: 909
Frames médios: 296.18


## 11. Resumo por categoria

In [14]:
print("RESUMO POR CATEGORIA\n")

for category in sorted(set(row["category"] for row in records)):
    subset = [row for row in records if row["category"] == category]

    labels = Counter(row["label"] for row in subset)
    attacks = Counter(row["attack_type"] for row in subset)
    media = Counter(row["media_type"] for row in subset)
    total_size = sum(row["file_size_mb"] for row in subset)

    print(category)
    print("  Amostras:", len(subset))
    print("  Label:", dict(labels))
    print("  Attack type:", dict(attacks))
    print("  Media:", dict(media))
    print(f"  Tamanho total: {total_size:.2f} MB")
    print()

RESUMO POR CATEGORIA

3D_paper_mask
  Amostras: 36
  Label: {'spoof': 36}
  Attack type: {'3d_paper_mask': 36}
  Media: {'video': 36}
  Tamanho total: 411.85 MB

Cutout_attacks
  Amostras: 15
  Label: {'spoof': 15}
  Attack type: {'cutout': 15}
  Media: {'video': 15}
  Tamanho total: 1117.33 MB

Latex_mask
  Amostras: 10
  Label: {'spoof': 10}
  Attack type: {'latex_mask': 10}
  Media: {'video': 10}
  Tamanho total: 719.85 MB

Replay_display_attacks
  Amostras: 10
  Label: {'spoof': 10}
  Attack type: {'replay_display': 10}
  Media: {'image': 5, 'video': 5}
  Tamanho total: 156.17 MB

Replay_mobile_attacks
  Amostras: 10
  Label: {'spoof': 10}
  Attack type: {'replay_mobile': 10}
  Media: {'video': 10}
  Tamanho total: 645.11 MB

Selfies
  Amostras: 24
  Label: {'bona_fide': 24}
  Attack type: {'none': 24}
  Media: {'image': 24}
  Tamanho total: 7.96 MB

Silicone_mask
  Amostras: 11
  Label: {'spoof': 11}
  Attack type: {'silicone_mask': 11}
  Media: {'video': 11}
  Tamanho total: 622.

## 12. Inspeção dos caminhos e nomes

Esta secção serve apenas para verificar se existe informação adicional
codificada nos nomes dos ficheiros ou subpastas. Nenhuma informação é
automaticamente interpretada como sujeito ou dispositivo.

In [15]:
for row in records[:100]:
    print(
        row["category"],
        "|",
        row["relative_path"]
    )

3D_paper_mask | 3D_paper_mask\133.mp4
3D_paper_mask | 3D_paper_mask\134.mp4
3D_paper_mask | 3D_paper_mask\135.mp4
3D_paper_mask | 3D_paper_mask\136.mp4
3D_paper_mask | 3D_paper_mask\137.mp4
3D_paper_mask | 3D_paper_mask\138.mp4
3D_paper_mask | 3D_paper_mask\139.mp4
3D_paper_mask | 3D_paper_mask\140.mp4
3D_paper_mask | 3D_paper_mask\141.mp4
3D_paper_mask | 3D_paper_mask\142.mp4
3D_paper_mask | 3D_paper_mask\143.mp4
3D_paper_mask | 3D_paper_mask\144.mp4
3D_paper_mask | 3D_paper_mask\145.mp4
3D_paper_mask | 3D_paper_mask\146.mp4
3D_paper_mask | 3D_paper_mask\147.mp4
3D_paper_mask | 3D_paper_mask\148.mp4
3D_paper_mask | 3D_paper_mask\149.mp4
3D_paper_mask | 3D_paper_mask\150.mp4
3D_paper_mask | 3D_paper_mask\151.mp4
3D_paper_mask | 3D_paper_mask\152.mp4
3D_paper_mask | 3D_paper_mask\153.mp4
3D_paper_mask | 3D_paper_mask\154.mp4
3D_paper_mask | 3D_paper_mask\155.mp4
3D_paper_mask | 3D_paper_mask\156.mp4
3D_paper_mask | 3D_paper_mask\157.mp4
3D_paper_mask | 3D_paper_mask\158.mp4
3D_paper_mas

## 13. Exportação do relatório CSV

In [18]:
REPORTS_PATH = DATASET_PATH.parent.parent / "Notebooks/results_datasets"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = REPORTS_PATH / "axondata_fas_dataset_report.csv"

fieldnames = [
    "filepath",
    "relative_path",
    "filename",
    "category",
    "label",
    "class_id",
    "attack_type",
    "media_type",
    "extension",
    "file_size_mb",
    "width",
    "height",
    "fps",
    "frame_count",
    "duration_seconds",
]

with OUTPUT_CSV.open(
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in records:
        writer.writerow(row)

print("Relatório exportado para:")
print(OUTPUT_CSV)

Relatório exportado para:
C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\axondata_fas_dataset_report.csv


## 14. Resumo automático

In [19]:
total = len(records)
bona_fide = sum(row["label"] == "bona_fide" for row in records)
spoof = sum(row["label"] == "spoof" for row in records)
images = sum(row["media_type"] == "image" for row in records)
videos = sum(row["media_type"] == "video" for row in records)

spoof_attack_types = {
    row["attack_type"]
    for row in records
    if row["label"] == "spoof"
}

print("RESUMO DO DATASET\n")
print("Total de amostras:", total)
print("Bona fide:", bona_fide)
print("Spoof:", spoof)
print("Imagens:", images)
print("Vídeos:", videos)
print("Tipos de ataque spoof identificados:", len(spoof_attack_types))

if total:
    print(f"Spoof (%): {spoof / total * 100:.2f}")
    print(f"Bona fide (%): {bona_fide / total * 100:.2f}")

RESUMO DO DATASET

Total de amostras: 149
Bona fide: 24
Spoof: 125
Imagens: 29
Vídeos: 120
Tipos de ataque spoof identificados: 8
Spoof (%): 83.89
Bona fide (%): 16.11


## Conclusão

A versão pública analisada do **AxonData Face Anti-Spoofing Dataset** contém
uma combinação de imagens e vídeos distribuídos por várias categorias de
apresentação.

A principal característica deste dataset é a diversidade de ataques,
incluindo ataques de replay, cutout e diferentes tipos de máscaras, como
máscaras de papel 3D, latex, silicone e tecido. Esta diversidade complementa
os datasets anteriormente analisados, que apresentam menor variedade de
ataques tridimensionais.

A versão pública apresenta um número relativamente reduzido de amostras e deve
ser considerada uma amostra do dataset disponibilizado pela AxonData. Além
disso, existe um desequilíbrio entre `bona_fide` e `spoof`, pelo que métricas
como `accuracy` não devem ser utilizadas isoladamente.

Outro aspeto importante é a diferença de modalidade entre algumas categorias:
as apresentações `bona_fide` disponíveis em `Selfies` são imagens, enquanto
muitos dos ataques são fornecidos em vídeo. Esta característica deve ser
considerada na definição de qualquer protocolo experimental para evitar que
um modelo aprenda diferenças relacionadas com o formato ou processo de
aquisição em vez de características efetivamente associadas a Presentation
Attack Detection.

A versão analisada não permite assumir de forma segura a existência de
identificadores consistentes de sujeitos, dispositivos ou splits oficiais sem
documentação adicional. Por esse motivo, essa informação não é inferida
automaticamente neste notebook.

Para a dissertação, este dataset será considerado principalmente como
**dataset auxiliar**, sendo especialmente interessante para avaliar robustez e
generalização perante tipos de ataques diferentes daqueles presentes nos
datasets académicos clássicos.